# M3L3 E11 — Support bot multiagente
### Módulo 3 · Lecture 3 · Sistemas Multiagente

## ¿Qué vas a aprender hoy?
- refactorizar el baseline con agentes especialistas.
- Conectar el concepto con M3L2.
- Leer código pequeño con explicación previa.
- Interpretar resultados y checks.


## ¿Qué necesitás saber antes?

Venís de M3L2 con LangChain, LCEL, PromptTemplate, RAG con FAISS y memoria conversacional. En M3L3 usamos esas piezas para coordinar varios agentes.

> **Sistema multiagente:** arquitectura donde varias unidades especializadas colaboran bajo una política de coordinación.


## Instalación e imports

En un notebook productivo podrías instalar `langchain`, `langchain-openai` y `faiss-cpu`. Aquí usamos Python estándar para que el foco sea el diseño multiagente y no la API key.


In [ ]:
from typing import Callable, TypedDict, Literal
from dataclasses import dataclass, field
import json

print("Setup listo: usamos Python estándar para que el notebook pueda correr sin API key.")


## Sección 1 — Refactor multiagente

Usamos el mismo benchmark de E10, pero separamos conocimiento, agentes y protocolo.

```text
query -> classify -> specialist_agent -> AgentResponse -> render
```


Cargamos `AgentResponse`, knowledge base por dominio y benchmark fijo.


In [ ]:
class AgentResponse(TypedDict):
    agent_name: str
    status: Literal["success", "needs_clarification", "out_of_scope"]
    answer: str
    confidence: Literal["high", "medium", "low"]
    needs_handoff: bool
    target_agent: str | None
    sources: list[str]

knowledge_base = {
    "hr": [
        "Vacaciones: cada empleado tiene 15 días hábiles por año.",
        "Beneficios: el seguro médico inicia el primer día de trabajo.",
        "Licencias: registrar el pedido en PeopleOps y avisar al manager.",
    ],
    "tech": [
        "VPN: reiniciar el cliente, validar MFA y abrir ticket si persiste.",
        "Contraseña: restablecer desde el portal de identidad.",
        "Notebook: reportar equipo dañado con número de serie.",
    ],
    "billing": [
        "Facturas: cargar comprobantes antes del día 25.",
        "Reembolsos: adjuntar recibo, monto y centro de costo.",
        "Pagos: se procesan los viernes por la tarde.",
    ],
}

KEYWORDS = {
    "hr": ["vacaciones", "beneficio", "seguro", "licencia"],
    "tech": ["vpn", "contraseña", "mfa", "notebook"],
    "billing": ["factura", "facturas", "reembolso", "pago", "recibo"],
}

def detect_domains(query: str) -> list[str]:
    text = query.lower()
    matches = [domain for domain, words in KEYWORDS.items() if any(word in text for word in words)]
    return matches or ["unknown"]

def retrieve(domain: str, query: str, k: int = 2) -> list[str]:
    docs = knowledge_base[domain]
    query_words = set(query.lower().replace("¿", "").replace("?", "").split())
    def score(doc: str) -> int:
        return sum(1 for word in query_words if word.strip(",.") in doc.lower())
    return sorted(docs, key=score, reverse=True)[:k]

benchmark_queries = [("vacaciones","hr"),("seguro","hr"),("vpn","tech"),("contraseña","tech"),("factura","billing"),("reembolso","billing"),("vacaciones y vpn","mixed"),("almuerzo","unknown"),("recibo","billing"),("notebook","tech")]


## Sección 2 — Orquestador final

Si detecta varios dominios, devuelve `mixed` y combina respuestas. Así el caso mixto ya no se fuerza a un solo agente.


In [ ]:
def specialist_agent(domain: str, query: str) -> AgentResponse:
    # TODO: responder con AgentResponse.
    return {"agent_name":domain,"status":"needs_clarification","answer":"TODO","confidence":"low","needs_handoff":False,"target_agent":None,"sources":[]}
def handle_query(query: str) -> dict:
    return {"predicted_domain":"unknown", "answer":"TODO"}


## Sección 3 — Tabla comparativa conceptual

Ejecutamos las mismas diez consultas. La comparación con E10 muestra el valor de separar responsabilidades.


In [ ]:
results = []
for q, expected in benchmark_queries:
    r = handle_query(q); ok = r["predicted_domain"] == expected; results.append(ok)
    print(f"{expected:8} | pred={r['predicted_domain']:8} | ok={ok} | {q}")
print("Accuracy multiagente:", sum(results), "/", len(results))


## Checks automáticos

Los checks verifican el contrato mínimo del ejercicio. En Starter pueden fallar hasta completar los TODOs; en Resolution deben pasar.


In [ ]:
def run_checks():
    assert handle_query("vacaciones")["predicted_domain"] == "hr"
    assert handle_query("vacaciones y vpn")["predicted_domain"] == "mixed"
    print("Checks E11 OK")
run_checks()


## ¿Qué aprendiste hoy?

- Refactorizar el baseline con agentes especialistas.
- Separar responsabilidades vuelve el sistema más auditable.
- Los contratos explícitos hacen que el orquestador dependa menos de texto libre.

## Próximo ejercicio

Continuá con el siguiente notebook de M3L3 para agregar una pieza más de coordinación multiagente.
